In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sentence_transformers import SentenceTransformer

df = pd.read_csv("../data/processed/strixhaven_clean.csv")

train, test = train_test_split(df, test_size=0.2, random_state=42)
print(f"Train: {len(train)} cards, Test: {len(test)} cards")

C:\Users\troyj\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: 211 cards, Test: 53 cards


In [2]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

X_train_emb = embedder.encode(train["text"].astype(str).tolist(), show_progress_bar=True)
X_test_emb = embedder.encode(test["text"].astype(str).tolist(), show_progress_bar=True)

print(f"Embedding shape: {X_train_emb.shape}")

C:\Users\troyj\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\troyj\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading w

Embedding shape: (211, 384)


In [3]:
model_emb = Ridge(alpha=5.0)
model_emb.fit(X_train_emb, train["win_rate"])

test_predictions_emb = model_emb.predict(X_test_emb)

rmse_emb = mean_squared_error(test["win_rate"], test_predictions_emb) ** 0.5
r2_emb = r2_score(test["win_rate"], test_predictions_emb)

print(f"Sentence embeddings + Ridge  ->  RMSE: {rmse_emb:.4f}   R²: {r2_emb:.4f}")

Sentence embeddings + Ridge  ->  RMSE: 0.0384   R²: 0.0840


In [4]:
results_by_alpha = []

for alpha in [0.5, 1.0, 5.0, 10.0, 20.0, 50.0]:
    m = Ridge(alpha=alpha)
    m.fit(X_train_emb, train["win_rate"])
    preds = m.predict(X_test_emb)
    rmse = mean_squared_error(test["win_rate"], preds) ** 0.5
    r2 = r2_score(test["win_rate"], preds)
    results_by_alpha.append({"alpha": alpha, "RMSE": rmse, "R2": r2})

alpha_results = pd.DataFrame(results_by_alpha)
print(alpha_results.to_string(index=False))

 alpha     RMSE       R2
   0.5 0.037730 0.113642
   1.0 0.037406 0.128785
   5.0 0.038356 0.083996
  10.0 0.038987 0.053609
  20.0 0.039512 0.027928
  50.0 0.039966 0.005478


In [5]:
final_results = pd.DataFrame({
    "Model": ["Mean baseline", "Text length only", "Mana value only", "TF-IDF + Ridge", "Sentence embeddings + Ridge"],
    "RMSE": [0.040371, 0.040035, 0.040247, 0.037706, rmse_emb],
    "R²": [-0.014777, 0.002052, -0.008564, 0.114800, r2_emb],
})
print(final_results.to_string(index=False))

                      Model     RMSE        R²
              Mean baseline 0.040371 -0.014777
           Text length only 0.040035  0.002052
            Mana value only 0.040247 -0.008564
             TF-IDF + Ridge 0.037706  0.114800
Sentence embeddings + Ridge 0.038356  0.083996


In [6]:
import joblib

joblib.dump(model_emb, "../models/embeddings_ridge_model.joblib")
print("Saved embeddings model to ../models/")

Saved embeddings model to ../models/
